# CLIP few-shot learning

In [1]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [2]:
import os

os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

In [3]:
# install CLIP if missing
try:
    import clip
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "git+https://github.com/openai/CLIP.git"], check=True)
    import clip

In [4]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

Drive mounted.
Using Colab cache for faster access to the 'ucf101-action-recognition' dataset.
ULAL_DATASET_ROOT = /kaggle/input/ucf101-action-recognition/
ULAL_OUTPUT_ROOT  = /kaggle/working/ucf101-processed/
ULAL_DATA_ROOT     = /content/UCF101
ULAL_DRIVE_PROJECT = /content/drive/MyDrive/apai
Setup complete — no config.py edit, no kernel restart.


In [5]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
Classes   : base=10 task1=5 task2=5


42

## **Load CLIP and data**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model, _ = clip.load("ViT-B/32", device=device)
clip_model.eval()

base_split = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
class_to_idx = {cls_name: i for i, cls_name in enumerate(cfg.SELECTED_CLASSES)}
num_classes = len(cfg.SELECTED_CLASSES)

# 16 training clips per class
max_samples = 16
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT, max_samples=max_samples)

base_train = UCF101Clips(os.path.join(cfg.BASE_ROOT, "train"), class_to_idx=class_to_idx)
base_val   = UCF101Clips(os.path.join(cfg.BASE_ROOT, "val"),   class_to_idx=class_to_idx)
base_test  = UCF101Clips(os.path.join(cfg.BASE_ROOT, "test"),  class_to_idx=class_to_idx)

batch_size = 8
train_loader = DataLoader(base_train, batch_size=batch_size, shuffle=True,  num_workers=0, drop_last=False)
val_loader   = DataLoader(base_val,   batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)
test_loader  = DataLoader(base_test,  batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)

## **CLIP with 16 clips per class: hybrid prototypes and linear probe**

In [ ]:
print("extracting CLIP features for train, val and test")
X_train, y_train = extract_clip_video_features(train_loader, clip_model, device)
X_val,   y_val   = extract_clip_video_features(val_loader,   clip_model, device)
X_test,  y_test  = extract_clip_video_features(test_loader,  clip_model, device)

text_prototypes = get_text_prototypes(cfg.SELECTED_CLASSES, clip_model, device)

# method 1: hybrid text + visual prototypes
val_acc_hybrid  = evaluate_hybrid_prototypes(X_train, y_train, X_val, y_val, text_prototypes, alpha=0.4)
test_acc_hybrid = evaluate_hybrid_prototypes(X_train, y_train, X_test, y_test, text_prototypes, alpha=0.4)

print(f"hybrid prototypes | val {val_acc_hybrid:.2f}% | test {test_acc_hybrid:.2f}%")

# method 2: linear probe (logistic regression)
clf = LogisticRegression(C=0.01, max_iter=1000, solver='lbfgs', multi_class='multinomial')
clf.fit(X_train, y_train)

val_acc_lp  = clf.score(X_val, y_val) * 100.0
test_acc_lp = clf.score(X_test, y_test) * 100.0

print(f"linear probe      | val {val_acc_lp:.2f}% | test {test_acc_lp:.2f}%")

## **k-shot sweep: ResNet50 + LSTM vs CLIP + MLP**

In [ ]:
import shutil

k_shots_list = [0, 1, 2, 4, 8, 16]
results = []

# test features are extracted once
print("extracting CLIP test features")
X_test_fixed, y_test_fixed = extract_clip_video_features(test_loader, clip_model, device)
test_ds_fixed = TensorDataset(torch.tensor(X_test_fixed, dtype=torch.float32), torch.tensor(y_test_fixed, dtype=torch.long))
test_feat_ld_fixed = DataLoader(test_ds_fixed, batch_size=32, shuffle=False)

for k in k_shots_list:
    print(f"\nk = {k} clips per class")

    if k == 0:   # placeholder, not run
        resnet_acc = 0.0
        mlp_acc = 0.0
    else:
        # rebuild the training set with k clips per class
        if os.path.exists(cfg.BASE_ROOT):
            shutil.rmtree(cfg.BASE_ROOT)

        preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT, max_samples=k)

        train_k = UCF101Clips(os.path.join(cfg.BASE_ROOT, "train"), class_to_idx=class_to_idx)
        train_loader_k = DataLoader(train_k, batch_size=min(batch_size, len(train_k)), shuffle=True, num_workers=0)

        # ResNet50 + LSTM (backbone frozen)
        set_seed(cfg.SEED)
        resnet_lstm = ResNet50LSTM(num_classes=num_classes).to(device)
        for param in resnet_lstm.backbone.parameters():
            param.requires_grad = False

        train_model(resnet_lstm, train_loader_k, val_loader, num_epochs=10, device=device)

        resnet_acc, _ = evaluate_model(resnet_lstm, test_loader, device=device)
        resnet_acc *= 100.0 if resnet_acc <= 1.0 else 1.0

        # CLIP features + MLP adapter
        X_train_k, y_train_k = extract_clip_video_features(train_loader_k, clip_model, device)
        train_ds_k = TensorDataset(torch.tensor(X_train_k, dtype=torch.float32), torch.tensor(y_train_k, dtype=torch.long))
        train_feat_ld_k = DataLoader(train_ds_k, batch_size=min(16, len(train_ds_k)), shuffle=True)

        mlp_model = CLIPMLPAdapter(in_dim=512, hidden_dim=256, num_classes=num_classes, dropout=0.3).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(mlp_model.parameters(), lr=1e-3, weight_decay=1e-2)

        for epoch in range(50):
            mlp_model.train()
            for x_b, y_b in train_feat_ld_k:
                x_b, y_b = x_b.to(device), y_b.to(device)
                optimizer.zero_grad()
                loss = criterion(mlp_model(x_b), y_b)
                loss.backward()
                optimizer.step()

        mlp_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x_b, y_b in test_feat_ld_fixed:
                x_b, y_b = x_b.to(device), y_b.to(device)
                preds = mlp_model(x_b).argmax(dim=-1)
                correct += (preds == y_b).sum().item()
                total += y_b.size(0)
        mlp_acc = (correct / total) * 100.0 if total > 0 else 0.0

    results.append({
        'k': k,
        'ResNet50+LSTM Test Acc (%)': resnet_acc,
        'CLIP+MLP Adapter Test Acc (%)': mlp_acc
    })

df_results = pd.DataFrame(results)
print("\ntest accuracy (%)")
print(df_results.to_string(index=False))


--- Preprocessing (group-disjoint) ---
To  : /content/UCF101/grouped_base
Classes: 10 | Train limit: 16

Processing split: train



Processing split: val



Processing split: test



Done. Clips ready at: /content/UCF101/grouped_base
Extracting CLIP embeddings for Train, Val, and Test splits...



[METHOD 1: Hybrid Prototypes] Val Acc  : 89.70%
[METHOD 1: Hybrid Prototypes] Test Acc : 90.09%

[METHOD 2: Linear Probe]     Val Acc  : 90.91%
[METHOD 2: Linear Probe]     Test Acc : 89.62%
Extracting fixed test features for CLIP adapters...


...

Epoch [6/10] | Train Acc: 0.9750 Loss: 0.7635 | Val Acc: 0.7818 Loss: 1.1675


Epoch [7/10] | Train Acc: 0.9750 Loss: 0.4897 | Val Acc: 0.7818 Loss: 0.9863


Epoch [8/10] | Train Acc: 1.0000 Loss: 0.3035 | Val Acc: 0.7758 Loss: 0.9079


Epoch [9/10] | Train Acc: 0.9938 Loss: 0.1891 | Val Acc: 0.8061 Loss: 0.8024


Epoch [10/10] | Train Acc: 1.0000 Loss: 0.1594 | Val Acc: 0.8121 Loss: 0.7423

         FINAL TEST ACCURACY COMPARISON MATRIX (%)       
 k  ResNet50+LSTM Test Acc (%)  CLIP+MLP Adapter Test Acc (%)
 0                    0.000000                       0.000000
 1                   17.924528                      75.000000
 2                   15.094340                      73.113208
 4                   36.320755                      74.056604
 8                   50.943396                      79.245283
16                   61.320755                      86.792453


## **Recap**

- Frozen CLIP ViT-B/32: every clip becomes one 512-d feature (mean of its 16 frames). 10 base classes, 212 test clips.
- With 16 clips per class: hybrid text + visual prototypes 90.1% (nothing trained), linear probe 89.6%.
- k-shot sweep: CLIP + MLP reaches 75.0% from 1 clip per class, against 17.9% for ResNet50 + LSTM; with 16 clips 86.8% against 61.3%.
- CLIP learns from very few clips, but its image encoder is large (about 88M parameters). The k = 0 row is a placeholder (not run).